# Customer Shopping Behavior — Data Preparation

**Dataset:** 3,900 customer transactions | 18 features  
**Stack:** Python (Pandas) → MySQL (SQLAlchemy) → SQL Analysis → Power BI  
**Goal:** Clean, engineer features, and load data into MySQL for downstream analysis.

---

## 1. Load & Explore Data

In [ ]:
import pandas as pd

df = pd.read_csv('Customer_Shopping_Behavior.csv')

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

## 2. Data Cleaning

In [ ]:
# Check for null values
df.isnull().sum()

In [ ]:
# Impute missing values in Review Rating with category median (if any exist)
df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(
    lambda x: x.fillna(x.median())
)

In [ ]:
# Confirm no nulls remain
df.isnull().sum()

In [ ]:
# Standardize column names to snake_case
df.columns = df.columns.str.lower().str.replace(' ', '_')
df = df.rename(columns={'purchase_amount_(usd)': 'purchase_amount'})

In [ ]:
df.columns

## 3. Feature Engineering

In [ ]:
# Bin customers into 4 equal age groups using quartile cuts
labels = ['Young Adult', 'Adult', 'Middle-aged', 'Senior']
df['age_group'] = pd.qcut(df['age'], q=4, labels=labels)

In [ ]:
df[['age', 'age_group']].head(10)

In [ ]:
# Map purchase frequency text to numeric days for quantitative analysis
frequency_mapping = {
    'Fortnightly': 14,
    'Weekly': 7,
    'Monthly': 30,
    'Quarterly': 90,
    'Bi-Weekly': 14,
    'Annually': 365,
    'Every 3 Months': 90
}
df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)

In [ ]:
df[['purchase_frequency_days', 'frequency_of_purchases']].head(10)

## 4. Data Quality Checks

In [ ]:
# Check if discount_applied and promo_code_used are redundant columns
df[['discount_applied', 'promo_code_used']].head(10)

In [ ]:
# Verify 100% identical before dropping to avoid accidental data loss
(df['discount_applied'] == df['promo_code_used']).all()

In [ ]:
# Drop redundant column
df = df.drop('promo_code_used', axis=1)

In [ ]:
# Final schema
print(f'Final dataset: {df.shape[0]} rows, {df.shape[1]} columns')
print(df.columns.tolist())

## 5. Load to MySQL

**Setup:** Create a `.env` file with your MySQL credentials before running this section:
```
MYSQL_USER=root
MYSQL_PASSWORD=your_password_here
```
See `.env.example` in this repo.

In [ ]:
!pip install pymysql python-dotenv

In [ ]:
import os
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()  # loads credentials from .env file

username = os.getenv('MYSQL_USER', 'root')
password = os.getenv('MYSQL_PASSWORD')  # set MYSQL_PASSWORD in .env file
host = 'localhost'
port = '3306'
database = 'customer_behavior'

engine = create_engine(f'mysql+pymysql://{username}:{password}@{host}:{port}/{database}')
print('Connection Successful!')

In [ ]:
df.to_sql(
    name='shopping_data',
    con=engine,
    if_exists='replace',
    index=False
)
print(f'Successfully loaded {len(df)} records into MySQL table: shopping_data')

## Summary

| Step | Action | Result |
|---|---|---|
| Load | Read CSV | 3,900 records, 18 columns |
| Clean | Null check + median imputation | 0 nulls in final dataset |
| Standardize | snake_case column naming | All columns consistent |
| Engineer | `age_group` + `purchase_frequency_days` | 2 new analytical columns |
| Deduplicate | Dropped `promo_code_used` (identical to `discount_applied`) | 19 clean columns |
| Export | Loaded to MySQL via SQLAlchemy | 3,900 rows → `shopping_data` table |

**Next step:** Run `mysql_customer_behavior_queries.sql` in MySQL Workbench to generate business insights.  
**Final output:** Open `customer_behavior_dashboard.pbix` in Power BI Desktop for the interactive dashboard.